# Talking to the profiler

This chapter is the book form of the packaged agent guide. The
guide ships inside the package — `onceover.agent_guide()` returns
it — so the version a caller reads in a session always matches
the build it calls. The two are condensed from the same source:
the reference lists in this chapter (the hint keys, the budgets)
are the guide's, verbatim, and a repository test fails if the two
diverge.

The order is the point. A profile is aimed before it is run: what
the caller declares about the table decides which passes engage
and what their findings mean. So the chapter starts with the
handover — what to declare, and how — then the call, then
reading the answer, then detector selection.


## 1. Hand over what you know before the call

**The outcome.** `target="<column>"` designates the response
column. A designated target engages five passes otherwise
skipped: `outcome_account` (the outcome's distribution),
`outcome_contrasts` (per-level event rates, or medians,
against the pooled outcome; absences stated as nulls),
`target_shapes` (each numeric predictor's relationship shape
to the target), `leakage` (a column that determines the
target or dwarfs every other relationship to it), and
`model_failures` (where a cross-fitted model of the target
fails). An unknown target goes to coverage, not an error: the
report states that the designation could not be used instead of
failing the run.

**Meaning and purpose.** Seven keys carry what the columns
mean and what the data is for. They ride the report's frame
account (`frame.declarations`); the `declarations` coverage
entry states what was checked and what was carried unchecked,
and nothing declared under them is silently dropped. `gloss`
({column: text, or {level: text}}): what a column means, in
your words — findings about the column state it. `units`
({column: unit}): magnitudes, stated through the findings'
`unit` field. `event`: the target level counting as the
event — the outcome contrasts and their nulls are stated
against it; without it they take a weaker per-level form.
`entity`: the panel entity column. `recorded_after`
([columns]): columns recorded after the outcome — leakage
checks them first as declared suspects, and the model screens
exclude them from their predictor pools. `purpose` (one
string) and `questions` ([strings]): what the data is for — a
finding answering a declared question (matched by the columns
the question names) is admitted ahead of an equal-score
unmatched finding, and the `questions` coverage entry states
how many findings answer each question, including zero: an
unanswered question is a result. Declaring any of these keys
engages the pointed mode: the outcome contrasts state their
measured nulls alongside their measured differences.

**Structure.** `roles` overrides a column's semantic
role; the vocabulary is `identifier`, `row_number`, `timestamp`,
`timestamp_part`, `dimension`, `flag`, `measure`, `text`. A role
outside the vocabulary is refused in `declarations` coverage;
valid declarations take in the frame section's evidence. `time`,
`ordered_axis`, and `level_order` are likewise applied or refused
there (a string `ordered_axis` needs a valid `level_order`).
`labels` gives a column a
display label for prose; evidence keeps raw names.

Hint keys: `time`, `ordered_axis`, `level_order`, `order_by`, `no_gaps`, `cadence`, `dimensions`, `measures`, `weights`, `roles`, `hierarchies`, `exclude_columns`, `labels`, `contrast`, `gloss`, `units`, `event`, `entity`, `recorded_after`, `purpose`, `questions`.

The expectations pass checks `order_by`, `no_gaps`, `cadence`,
`hierarchies`, and `dimensions`/`measures` exactly; checked
declarations tag their evidence `reference_source="declared"`,
and unusable entries go to coverage, not errors. `cadence`
values are `"daily"`, `"business_day"`, `"weekly"`, `"monthly"`,
or a step count; `hierarchies` are level columns, finest to
coarsest. Instructions to the run: `time`, `ordered_axis` (a
non-date ordering column), `level_order` ({column: [levels in
order]}, never inferred), `exclude_columns` (dropped pre-run),
`contrast` (`{"column": name, "value": scalar}`; a boolean
column needs no `value`) profiles one subset against the rest;
`weights` ([columns]) makes the hotspots pass also judge cells
by the share of each declared weight their rows carry, not
just by rows. Other unknown keys raise an error.

In [1]:
from datetime import date, timedelta

import polars as pl

import onceover

n = 240
bookings = pl.DataFrame(
    {
        "booking_id": list(range(1, n + 1)),
        "arrival_date": [
            date(2024, 1, 1) + timedelta(days=i % 180) for i in range(n)
        ],
        "lead_time": [(i * 37) % 200 for i in range(n)],
        "adr": [40.0 + float((i * 53) % 160) for i in range(n)],
        "hotel": ["Resort" if i % 3 == 0 else "City" for i in range(n)],
        "total_nights": [1 + i % 7 for i in range(n)],
        "is_canceled": [1 if (i * 7) % 10 < 3 else 0 for i in range(n)],
    }
)

report = onceover.profile(
    bookings,
    target="is_canceled",
    hints={
        "time": "arrival_date",
        "event": 1,
        "units": {"adr": "euros"},
        "questions": ["what drives cancellations?"],
        "labels": {
            "adr": "average daily rate",
            "lead_time": "days booked ahead",
        },
    },
)
print(report.to_digest())

profile digest: 240 rows x 7 columns, 15 findings, schema 0.8
What a row is:
the grain is 7 total_nights × 180 arrival_date: (total_nights, arrival_date) jointly identify rows in 240 rows — the cross is not full (1,260 combinations, 1,020 absent)
time axis — declared by hints.time: the column 'arrival_date', 2024-01-01 to 2024-06-28, in 1 day steps (180 distinct values)
What relates to the target (is_canceled):
The designated outcome 'is_canceled' is numeric: median 0, mean 0.3, middle half 0 to 1, range 0 to 1, over 240 values. [concentration-730b9a3296617174]
No difference in the median of 'is_canceled' across 'total_nights' was measured: '1' 0 (35 values), '2' 0 (35 values), '3' 0 (34 values), '4' 0 (34 values), '5' 0 (34 values), '6' 0 (34 values), '7' 0 (34 values), against the pooled median 0. [dependencies-b61658aa3c377b7e]
No difference in the median of 'is_canceled' across 'day of week of arrival_date' was measured: 'Sunday' 0 (33 values), 'Monday' 0 (35 values), 'Tuesday' 0 (

## 2. The call and the budgets

`profile()` also accepts a LazyFrame or a file path, sampled
above 200,000 rows unless `exact=True`: the `source` section
states the basis.

Budgets: `quick`, `standard`, `deep`.

`quick`: summaries and integrity checks. `standard` (default):
the default set, capped. `deep`: wider caps, deeper search.
`deadline_seconds=` bounds the run. Bad budgets, detectors, hint
keys, or hint values raise errors listing the valid values — the
last cell of this chapter shows one.

`onceover.capabilities()` returns the machine-readable companion
to this chapter: the budgets, the hint keys, and one entry per
registered detector with its membership, trigger, and purpose,
derived from the registry on every call.


In [2]:
import json

print(json.dumps(onceover.capabilities(), indent=1))


{
 "version": "0.1.6",
 "schema_version": "0.8",
 "budgets": [
  "quick",
  "standard",
  "deep"
 ],
 "hint_keys": [
  "time",
  "ordered_axis",
  "level_order",
  "order_by",
  "no_gaps",
  "cadence",
  "dimensions",
  "measures",
  "weights",
  "roles",
  "hierarchies",
  "exclude_columns",
  "labels",
  "contrast",
  "gloss",
  "units",
  "event",
  "entity",
  "recorded_after",
  "purpose",
  "questions"
 ],
 "detectors": [
  {
   "id": "column_profile",
   "membership": "default_set",
   "trigger": "always",
   "purpose": "Column summaries, role suggestions, missingness, concentration, string formats"
  },
  {
   "id": "row_duplicates",
   "membership": "default_set",
   "trigger": "always",
   "purpose": "Repeated rows under the stated comparison and missing-value policy"
  },
  {
   "id": "keys",
   "membership": "default_set",
   "trigger": "always",
   "purpose": "Single-column, composite, and nearly unique keys"
  },
  {
   "id": "pivot",
   "membership": "on_request",
   "tr

## 3. Reading the answer

The digest, `report.to_digest()`, is a briefing in sections by
reader question — what a row is (the grain and the time axis),
what is wrong, what is redundant, what relates to the target,
other relationships, anomalies, and not examined — one line
per fact, each line naming the finding that speaks for it;
`report.finding(id)` expands one.
`report.coverage` records each pass as run, skipped, capped, or
ineligible; an empty findings list with skipped or capped
coverage means the table was not fully examined, not clean data.

A finding's key is its (`category`, `type`) pair;
`report.select(category=..., type=..., column=..., target=...)`
also filters on facet fields. The `claim` is a statement with
evidence attached: read its `metrics` payload and `basis`
(`measured`, `estimated`, `inferred`) before quoting it.
`interestingness` is the selection score that orders the
report — the mean of the detector's own score and a ledger
evidence score; it is not a probability. Verify before acting: `report.verify(df)`
recounts the statements; `report.rows(id, df)` returns one
statement's rows.


In [3]:
target_findings = report.select(target="is_canceled")
print(len(target_findings), "findings about the outcome")
for finding in target_findings[:3]:
    print("-", finding["claim"])


4 findings about the outcome
- No difference in the median of 'is_canceled' across 'total_nights' was measured: '1' 0 (35 values), '2' 0 (35 values), '3' 0 (34 values), '4' 0 (34 values), '5' 0 (34 values), '6' 0 (34 values), '7' 0 (34 values), against the pooled median 0.
- No difference in the median of 'is_canceled' across 'day of week of arrival_date' was measured: 'Sunday' 0 (33 values), 'Monday' 0 (35 values), 'Tuesday' 0 (35 values), 'Wednesday' 0 (35 values), 'Thursday' 0 (35 values), 'Friday' 0 (34 values), 'Saturday' 0 (33 values), against the pooled median 0.
- No difference in the median of 'is_canceled' across 'hotel' was measured: 'Resort' 0 (80 values), 'City' 0 (160 values), against the pooled median 0.


## 4. Detector selection

The default set runs most of the registered detectors;
`detectors=[...]` restricts a run to a named subset.
`capabilities()` lists each detector's membership (`default_set`
or `on_request`), trigger (`always`, `hints`, or `target`), and
purpose. Selection is unnecessary at first: profile once with
the handover in place, read the coverage to see what ran, and
restrict only when a narrower question follows.

An unknown detector ID is an error, and the error lists the
registered IDs:


In [4]:
try:
    onceover.profile(bookings, detectors=["leakge"])
except ValueError as error:
    print(error)


unknown detector: leakge (registered detectors: association_rules, boundary_mass, breaks, column_profile, content_anomalies, covering_rules, date_gaps, decision_tables, digit_fingerprints, discontinuities, distribution_change, drift, duplicate_encodings, entity_fingerprints, expectations, functional_dependencies, granularity, group_contrast, hotspots, identities, interactions, keys, leakage, level_contrasts, missingness, model_failures, numeric_dependence, order_constraints, ordered_axis, ordered_partitions, outcome_account, outcome_contrasts, outliers, panel_coverage, period_associations, pivot, predictability, regimes, row_clusters, row_consistency, row_duplicates, sentinels, series, slice_shifts, sortedness, target_shapes)
